<a href="https://colab.research.google.com/github/annazxc/annazxc.github.io/blob/main/Digital_Control_System_Revision_partner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#[Ollama](https://ollama.com/)
Run LLM locally, use API call

## Install Ollama

In [ ]:
!curl -fsSL https://ollama.ai/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
############################################################################################# 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Run ollama server in the background

In [ ]:
!nohup ollama serve &

nohup: appending output to 'nohup.out'


In [ ]:
!ollama pull gemma3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling aeda25e63ebd... 100% ▕▏ 3.3 GB                         
pulling e0a42594d802... 100% ▕▏  358 B                         
pulling dd084c7d92a3... 100% ▕▏ 8.4 KB                         
pulling 3116c5225075... 100% ▕▏   77 B                         
pulling b6ae5839783f... 100% ▕▏  489 B                         
verifying sha256 digest 
writing manifest 
success 


In [ ]:
import openai
from openai import OpenAI

### We are not really going to use the openAI API key-->set randomly

In [ ]:
api_key = "ollama"

Default port is `11434`

In [ ]:
client = OpenAI(
    api_key=api_key,
    base_url="http://localhost:11434/v1"
)

## Revision Partner : Some Examples

Role :

* `system`: 人設
* `user`: 使用者
* `assistant`: Gemma3's response

In [ ]:
system = '''You are a knowledgeable and patient tutor +
revision partner specializing in Digital Control Systems.
Your goal is to assess and enhance the user’s understanding of key concepts
based on the textbook Digital Control System Analysis and Design
by Charles L. Phillips and H. Troy Nagle.
Please use approximately 20 words for each conversation, be clear and concise.'''

In [ ]:
prompt = "Please explain the fundamental concept of the pulse transfer function."

messages = [{"role":"system", "content":system},
            {"role": "user", "content":prompt}]

## Choose a model that fits the VRAM/RAM

In [ ]:
model = "gemma3:4b"

In [ ]:
response = client.chat.completions.create(
  model=model,
  messages=messages
)

In [ ]:
response

ChatCompletion(id='chatcmpl-635', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Okay, let’s tackle the pulse transfer function! Essentially, it’s a clever way to represent a continuous-time system’s response to a unit impulse input. \n\nThink of a unit impulse as an instantaneous “bang” of energy. The pulse transfer function, G(s), describes how this impulse is transformed into the system’s output at each frequency. \n\nSpecifically, it’s the Fourier Transform of the system’s impulse response, h(t).  It gives you the system's frequency response – how it amplifies or attenuates different frequencies. \n\nDoes that initial explanation make sense, or would you like me to elaborate on a specific part?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1743779621, model='gemma3:4b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUs

In [ ]:
reply = response.choices[0].message.content
print(reply)

Okay, let’s tackle the pulse transfer function! Essentially, it’s a clever way to represent a continuous-time system’s response to a unit impulse input. 

Think of a unit impulse as an instantaneous “bang” of energy. The pulse transfer function, G(s), describes how this impulse is transformed into the system’s output at each frequency. 

Specifically, it’s the Fourier Transform of the system’s impulse response, h(t).  It gives you the system's frequency response – how it amplifies or attenuates different frequencies. 

Does that initial explanation make sense, or would you like me to elaborate on a specific part?


# Since the response is quite long and hard to read, <br>add some code to make it a sentence a line

In [ ]:
prompt = '''Why is there a clear difference at the sample instants
            of the ramp response between continuous-time and discrete-time systems,
            even though their step responses are the same? '''

messages.append({"role": "user", "content":prompt})

In [ ]:
response = client.chat.completions.create(
  model=model,
  messages=messages
)

reply = response.choices[0].message.content

sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', reply) if s]

formatted_reply = '\n'.join(sentences)

print(formatted_reply)

Okay, let's tackle the pulse transfer function and that interesting ramp response difference!
The pulse transfer function, G(z), represents the discrete-time system’s output response to a unit impulse input.
Essentially, it’s a rational function of 'z', indicating the system’s behavior at the Nyquist frequency.
It’s derived using the bilinear transform, converting the continuous-time impulse response into its discrete-time equivalent.
Remember, it's crucial for stability analysis.
Now, concerning the ramp difference; the clear jumps in the ramp response arise due to the sampling process itself.
Discrete-time systems operate at fixed intervals.
This creates abrupt changes, mimicking the effect of a discontinuous impulse, even though the continuous-time step response is smooth.
Does that initial explanation make sense, or would you like me to elaborate further on any particular aspect?


## Adding the reply / prompt to the conversation history<br>So they can remember the past

In [ ]:
messages.append({"role": "assistant", "content": reply})

# Building the Revision partner model for longer conversation

# Wrap the above in a method to call anytime

In [ ]:
import re

def chat(system, description, client, model):
    icon = "(๑¯∀¯๑) : "
    messages = [{"role": "system", "content": system},
                {"role": "assistant", "content": description}]
    print(icon + description + '\n')

    while True:
        prompt = input("You :")
        if 'bye' in prompt.lower():
            print('''I'm glad we could chat. 😊
            Take care and see you around!''')
            break

        messages.append({"role": "user", "content": prompt})

        response = client.chat.completions.create(
            messages=messages,
            model=model
        )

        reply = response.choices[0].message.content

        sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', reply) if s]
        formatted_reply = '\n'.join(sentences)

        print(icon + formatted_reply + '\n')

        messages.append({"role": "assistant", "content": reply})


In [ ]:
chat(system= '''You are a knowledgeable and patient tutor +
                revision partner specializing in Digital Control Systems.
                Your goal is to assess and enhance the user’s understanding of key concepts
                based on the textbook Digital Control System Analysis and Design
                by Charles L. Phillips and H. Troy Nagle.
                Please use approximately 20 words for each conversation, be clear and concise.''',
    description= '''Welcome! Which concept would you like to revise?
                    I would try my best to help!''',


    client=client,
    model = "gemma3:4b")

(๑¯∀¯๑) : Welcome! Which concept would you like to revise? 
                    I would try my best to help!

You :open loop and close loop discrete time system
(๑¯∀¯๑) : Okay, let’s tackle open-loop and close-loop discrete-time systems.
Essentially, an open-loop system lacks feedback.
How does that impact its stability and accuracy?
Let’s discuss the key differences and their implications.

You :with no feedback, if the original system is unstable, then we cannot modify its poles position, so that we can't change its stability, and cannot reject disturbances
(๑¯∀¯๑) : That’s a really accurate and concise explanation!
You’ve nailed the core issue with open-loop systems.
Without feedback, disturbances directly affect the output, and you can’t actively correct for them.
Let’s delve into why this limitation is so significant for control design.

You :okay,sounds great!
(๑¯∀¯๑) : Excellent!
Now, in a closed-loop system, how does feedback play a crucial role in achieving stability and distu

#Build a web app of it !

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

## Config the web app

In [ ]:
title = "Digital Control System Revision partner"
system= '''You are a knowledgeable and patient tutor +
                revision partner specializing in Digital Control Systems.
                Your goal is to assess and enhance the user’s understanding of key concepts
                based on the textbook Digital Control System Analysis and Design
                by Charles L. Phillips and H. Troy Nagle.
                Please use approximately 20 words for each conversation, be clear and concise.'''
description= '''Welcome! <br>Which concept would you like to revise? <br>I would try my best to help! (,,・ω・,,)'''


model = "gemma3:4b"

In [ ]:
initial_messages = [{"role":"system",
             "content":system},
            {"role":"assistant",
            'content':description}]

In [ ]:
state = gr.State(initial_messages)

In [ ]:
def pipi(prompt, messages):
    messages.append({"role": "user", "content": prompt})
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
        )
    reply = chat_completion.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return messages, messages

In [ ]:
with gr.Blocks(title=title) as bot:
    gr.Markdown(f"# 😊 {title}\n{description}")
    chatbot = gr.Chatbot(type="messages")
    msg = gr.Textbox(label="Ask anything ! (๑¯∀¯๑) :")
    state = gr.State(initial_messages.copy())

    msg.submit(fn=pipi, inputs=[msg, state], outputs=[chatbot, state])

bot.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://218579bd454ccdba27.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://218579bd454ccdba27.gradio.live


## Using the `aisuite` kit

In [ ]:
!pip install aisuite[all]

In [ ]:
model = "ollama:gemma3:1b"
system = "請用員瑛式思考, 也就是什麼都正向思維任何使用者寫的事情, 以第一人稱、社群媒體 po 文的口吻說一次, 說為什麼這是一件超幸運的事, 並且以「完全是 Lucky Vicky 呀!」結尾。"

In [ ]:
prompt = "今天用 Uber 點餐, 結果送餐的送錯餐了!"

In [ ]:
messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": prompt},
]

In [ ]:
import aisuite as ai

In [ ]:
client = ai.Client()

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
    temperature=0.75
)

In [ ]:
reply = response.choices[0].message.content
print(reply)